# 检查RouterV7是否正常工作

In [4]:
from utils import RouterV7
import transformers
import torch
import torch.nn as nn
import torch.nn.functional as F

In [8]:
tokenizer = transformers.AutoTokenizer.from_pretrained("/mnt/workspace/workgroup/qianli.myf/models/models_from_oss/huggingface/meta-AI/llama-2-7b-hf")
tokenizer.pad_token = tokenizer.eos_token

config = transformers.AutoConfig.from_pretrained("/mnt/workspace/workgroup/qianli.myf/models/models_from_oss/huggingface/meta-AI/llama-2-7b-hf")
embedding_layer = nn.Embedding(config.vocab_size, config.hidden_size, config.pad_token_id)
router = RouterV7(config, 8)

In [9]:
prompt = "I love you!"
inputs = tokenizer(prompt, padding="max_length", max_length=4096, return_tensors="pt")
ids = inputs.input_ids
mask = inputs.attention_mask

In [12]:
router(embedding_layer(ids), mask.float()).shape

torch.Size([1, 4096, 8])

# 检查LlamaMoEV7ForCausalLM是否正常工作

In [1]:
from transformers import AutoModel, LlamaForCausalLM, LlamaTokenizer, GenerationConfig, LlamaModel, AutoConfig, LlamaMoEV7ForCausalLM

import torch

# base_model_path = "/mnt/workspace/workgroup/qianli.myf/models/models_from_oss/huggingface/meta-AI/Llama-2-13b-hf"
base_model_path = "/mnt/workspace/workgroup/qianli.myf/models/models_from_oss/huggingface/meta-AI/llama-2-7b-hf"
# base_model_path = "/mnt/workspace/workgroup/qianli.myf/models/models_from_oss/huggingface/meta-AI/llama-2-70b-hf"
# base_model_path = "/mnt/workspace/workgroup/qianli.myf/projects/shabi/search_LLMs/tars_eval/lm-evaluation-harness/model-73.94_arc0-8+miscon0-8"
# config = LoraConfig(r=4, lora_alpha=2, target_modules=["gate_proj", "down_proj", "up_proj"], lora_dropout=0.05,)
print(f"loading base model from {base_model_path}")
config = AutoConfig.from_pretrained(base_model_path)
model = LlamaMoEV7ForCausalLM.from_pretrained(base_model_path, device_map="auto", config=config, expert_nums=4)

print(model)

loading base model from /mnt/workspace/workgroup/qianli.myf/models/models_from_oss/huggingface/meta-AI/llama-2-7b-hf


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

LlamaMoEV7ForCausalLM(
  (model): LlamaModelMoEV7Base(
    (embed_tokens): Embedding(32000, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm()
        (post_attention_layernorm): LlamaRMSNorm()
      )
    )
    (norm): LlamaRMSNor

In [3]:
"router" in dir(model.model)

True

In [6]:
model.model.router = RouterV7(model.config, 4)
print(model)

LlamaMoEV7ForCausalLM(
  (model): LlamaModelMoEV7Base(
    (embed_tokens): Embedding(32000, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm()
        (post_attention_layernorm): LlamaRMSNorm()
      )
    )
    (norm): LlamaRMSNor

# 尝试搭建PeftModel

In [1]:
from peft import PeftModel, LoraConfig
from utils import RouterV7

In [2]:
from peft import LoraModel, LoraConfig, MoELoraModel
from transformers import AutoModel, LlamaForCausalLM, LlamaTokenizer, GenerationConfig, LlamaModel, AutoConfig, LlamaMoEV7ForCausalLM

import torch

base_model_path = "/mnt/workspace/workgroup/qianli.myf/models/models_from_oss/huggingface/meta-AI/llama-2-70b-hf"
print(f"loading base model from {base_model_path}")
config = AutoConfig.from_pretrained(base_model_path)
model = LlamaMoEV7ForCausalLM.from_pretrained(base_model_path, device_map="auto", config=config, expert_nums=4)
tokenizer = LlamaTokenizer.from_pretrained(base_model_path)
tokenizer.pad_token = tokenizer.eos_token

print(model)

loading base model from /mnt/workspace/workgroup/qianli.myf/models/models_from_oss/huggingface/meta-AI/llama-2-70b-hf


Loading checkpoint shards:   0%|          | 0/15 [00:00<?, ?it/s]

LlamaMoEV7ForCausalLM(
  (model): LlamaModelMoEV7Base(
    (embed_tokens): Embedding(32000, 8192, padding_idx=0)
    (layers): ModuleList(
      (0-79): 80 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=8192, out_features=8192, bias=False)
          (k_proj): Linear(in_features=8192, out_features=1024, bias=False)
          (v_proj): Linear(in_features=8192, out_features=1024, bias=False)
          (o_proj): Linear(in_features=8192, out_features=8192, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=8192, out_features=28672, bias=False)
          (up_proj): Linear(in_features=8192, out_features=28672, bias=False)
          (down_proj): Linear(in_features=28672, out_features=8192, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm()
        (post_attention_layernorm): LlamaRMSNorm()
      )
    )
    (nor

In [3]:
peft_model = PeftModel.from_pretrained(model, "/mnt/workspace/workgroup/qianli.myf/data/trained_models/icbu-tars-0922_llama-2-70B_llm2_miscon_DDP-qlora-adaptor/checkpoint-105/adapter_model", "miscon")

In [4]:
peft_model.model.model.router = RouterV7(model.config, 2).to(peft_model.model.model.embed_tokens.weight.device)

In [5]:
peft_model.load_adapter("/mnt/workspace/workgroup/qianli.myf/data/trained_models/icbu-tars-0906_llama-2-70b_llm2_samantha-1.1-6.5K_DDP-qlora-adaptor/checkpoint-125/adapter_model", "samantha")

_IncompatibleKeys(missing_keys=['base_model.model.model.embed_tokens.weight', 'base_model.model.model.layers.0.self_attn.q_proj.weight', 'base_model.model.model.layers.0.self_attn.q_proj.lora_A.miscon.weight', 'base_model.model.model.layers.0.self_attn.q_proj.lora_B.miscon.weight', 'base_model.model.model.layers.0.self_attn.k_proj.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_A.miscon.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_B.miscon.weight', 'base_model.model.model.layers.0.self_attn.v_proj.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_A.miscon.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_B.miscon.weight', 'base_model.model.model.layers.0.self_attn.o_proj.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_A.miscon.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_B.miscon.weight', 'base_model.model.model.layers.0.mlp.gate_proj.weight', 'base_model.model.model.layers.0.mlp.gate_proj

In [6]:
print(peft_model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaMoEV7ForCausalLM(
      (model): LlamaModelMoEV7Base(
        (embed_tokens): Embedding(32000, 8192, padding_idx=0)
        (layers): ModuleList(
          (0-79): 80 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): Linear(
                in_features=8192, out_features=8192, bias=False
                (lora_dropout): ModuleDict(
                  (miscon): Dropout(p=0.05, inplace=False)
                  (samantha): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (miscon): Linear(in_features=8192, out_features=64, bias=False)
                  (samantha): Linear(in_features=8192, out_features=64, bias=False)
                )
                (lora_B): ModuleDict(
                  (miscon): Linear(in_features=64, out_features=8192, bias=False)
                  (samantha): Linear(in_features=64, out_features=8192, bias=False)


# 检验是否可以正常前向

In [7]:
prompt = "I hate you!"
def evaluate(
     prompt,
     input=None,
     temperature=0.1,
     top_p=0.75,
     top_k=40,
     max_new_tokens=256,
     **kwargs,
 ):
    device = "cuda"
    model = kwargs.pop("model", None)
    tokenizer.pad_token_id = tokenizer.eos_token_id
    inputs = tokenizer(prompt, return_tensors="pt", padding="max_length", max_length=4096)
    input_ids = inputs["input_ids"].to(device)
    data_dict = {
        'input_ids': input_ids,
        'attention_mask': inputs.attention_mask.bool(),
    }
    print([d.shape for d in data_dict.values()])
    print(input_ids.shape)
    generation_config = GenerationConfig(
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        no_repeat_ngram_size=3,
        **kwargs,
    )

    with torch.no_grad():
        generation_output = model.generate(
            **data_dict,
            generation_config=generation_config,
            return_dict_in_generate=True,
            output_scores=True,
            max_new_tokens=max_new_tokens,
        )

In [9]:
print(evaluate(prompt, model=model))

[torch.Size([1, 4096]), torch.Size([1, 4096])]
torch.Size([1, 4096])
torch.Size([1, 4096, 8192]) torch.Size([1, 4096])
cuda:0
calc gating distribution


This is a friendly reminder - the current text generation call will exceed the model's predefined maximum length (4096). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.


torch.Size([1, 1, 8192]) torch.Size([1, 4097])
cuda:0
calc gating distribution
torch.Size([1, 1, 8192]) torch.Size([1, 4098])
cuda:0
calc gating distribution
torch.Size([1, 1, 8192]) torch.Size([1, 4099])
cuda:0
calc gating distribution
torch.Size([1, 1, 8192]) torch.Size([1, 4100])
cuda:0
calc gating distribution
torch.Size([1, 1, 8192]) torch.Size([1, 4101])
cuda:0
calc gating distribution
torch.Size([1, 1, 8192]) torch.Size([1, 4102])
cuda:0
calc gating distribution
torch.Size([1, 1, 8192]) torch.Size([1, 4103])
cuda:0
calc gating distribution
torch.Size([1, 1, 8192]) torch.Size([1, 4104])
cuda:0
calc gating distribution
torch.Size([1, 1, 8192]) torch.Size([1, 4105])
cuda:0
calc gating distribution
torch.Size([1, 1, 8192]) torch.Size([1, 4106])
cuda:0
calc gating distribution
torch.Size([1, 1, 8192]) torch.Size([1, 4107])
cuda:0
calc gating distribution
torch.Size([1, 1, 8192]) torch.Size([1, 4108])
cuda:0
calc gating distribution
torch.Size([1, 1, 8192]) torch.Size([1, 4109])
cuda:

In [8]:
for n,p in peft_model.named_parameters():
    if "router" in n:
        print(p.device)

cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
